## Feature Selection with Financial Indicators
- In this notebook we will explore the effect of Debtor/ Tuition / Scholarship information in Semester 1 Model. We were not sure when these data was  obtained so we refrained using them in the model because of a possible leak. Still, we will see how the data affects the model.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from src.train import build_pipeline, cross_validate_model
from src.data import load_data, split_data
from src import config

In [3]:
features = config.SEMESTER_1_MODEL_FEATURES +["Debtor", "Tuition fees up to date", "Scholarship holder"]


In [4]:
df = load_data(path = config.DATA_PATH, target= config.TARGET)

In [5]:
X_train_fin, X_test_fin, y_train_fin, y_test_fin = split_data(df = df,
                                              features= features,
                                              target = config.TARGET,
                                              random_state= config.RANDOM_STATE,
                                              test_size= config.TEST_SIZE
                                              )

X_train_reg, X_test_reg, y_train_reg, y_test_reg = split_data(df = df,
                                              features= config.SEMESTER_1_MODEL_FEATURES,
                                              target = config.TARGET,
                                              random_state= config.RANDOM_STATE,
                                              test_size= config.TEST_SIZE
                                              )

In [6]:
numeric_features_fin = [f for f in config.NUMERICAL_FEATURES if f in X_train_fin.columns]
categoric_features_fin = [f for f in config.CATEGORICAL_FEATURES if f in X_train_fin.columns]

numeric_features_reg = [f for f in config.NUMERICAL_FEATURES if f in X_train_reg.columns]
categoric_features_reg= [f for f in config.CATEGORICAL_FEATURES if f in X_train_reg.columns]

In [7]:
model_fin= build_pipeline(categorical=categoric_features_fin, numerical=numeric_features_fin)
cross_validate_model(name='Semester 1 Financial Experiment',
                     model=model_fin,
                     random_state=config.RANDOM_STATE,
                     X=X_train_fin,
                     y=y_train_fin
                     )
model_reg = build_pipeline(categorical=categoric_features_reg, numerical=numeric_features_reg)
cross_validate_model(name='Semester 1 Regular',
                     model=model_reg,
                     random_state=config.RANDOM_STATE,
                     X=X_train_reg,
                     y=y_train_reg
                     )

Semester 1 Financial Experiment recall: 0.845 +/- 0.028
Semester 1 Financial Experiment f1_score: 0.850 +/- 0.024
Semester 1 Regular recall: 0.817 +/- 0.033
Semester 1 Regular f1_score: 0.816 +/- 0.025


### Findings

- Accidentally, test set was reviewed. However decision was made based on CV results only. 
- The result is CV recall 0.817 → 0.845 and F1 0.816 → 0.850
- Moderate gain we obtained by introducing possibly leaking features is not worth to take the risk.
- If the school can confirm that payment and scholarship status are known by the end of semester 1, these features could be added for a modest gain (+0.03 recall, about the size of the CV spread)


## Students with zero first-semester evaluations
- We will now explore the effect of students with zero first semester evaluations to our model. We want to see how many of these students would drop out


In [8]:
zero_eval_filter = X_train_reg['Curricular units 1st sem (evaluations)']==0
zero_eval_indexes = X_train_reg[zero_eval_filter].index
non_zero_eval_filter = X_train_reg['Curricular units 1st sem (evaluations)']!=0
non_zero_eval_indexes = X_train_reg[non_zero_eval_filter].index


In [9]:
num_of_zero_evals = len(X_train_reg[zero_eval_filter])
print(f'{num_of_zero_evals} students out of {len(X_train_reg)} have zero evaluations in our training set.')

251 students out of 2904 have zero evaluations in our training set.


In [10]:
y_train_reg.value_counts()

Target
0    1767
1    1137
Name: count, dtype: int64

In [11]:
y_train_reg[zero_eval_indexes].value_counts()

Target
1    196
0     55
Name: count, dtype: int64

In [12]:
y_train_reg[zero_eval_indexes].value_counts(normalize = True)

Target
1    0.780876
0    0.219124
Name: proportion, dtype: float64

In [13]:
y_train_reg[non_zero_eval_indexes].value_counts()

Target
0    1712
1     941
Name: count, dtype: int64

In [14]:
print(y_train_reg.value_counts(normalize = True))
print(y_train_reg[non_zero_eval_indexes].value_counts(normalize = True))

Target
0    0.608471
1    0.391529
Name: proportion, dtype: float64
Target
0    0.645307
1    0.354693
Name: proportion, dtype: float64


- We had 1137 dropouts in our training data before filtering the students with zero evaluations. 
- After the screening number dropped to 941. Out of 251 these students 196 had dropped out.
- Before the screening, 39% of the students are dropouts, afterwards 35% are dropouts

In [15]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=config.RANDOM_STATE)
y_cv_pred = cross_val_predict(model_reg,X_train_reg,y_train_reg, cv = cv )

In [16]:
y_pred_series = pd.Series(y_cv_pred, index = X_train_reg.index)

In [17]:
y_train_zero_evals = y_train_reg[zero_eval_indexes]
y_pred_zero_evals = y_pred_series[zero_eval_indexes]

In [18]:
print(confusion_matrix(y_train_zero_evals, y_pred_zero_evals))

[[  6  49]
 [  6 190]]


### Findings
- Out of 196 dropouts, we missed 6 of them with our model.
- 49 of 55 graduates were flagged as dropout by the model. We did not expect this many students with zero first semester evaluations to graduate.
- Model already handles students with zero evaluations, we do not need to rearrange our data or features